## Speech recognition with Conformer-CTC

This demo shows how to use our project. It will download a trained model, transcribe your own audio, and calculate CER and WER with reference transcriptions, if they are available.

### 1. Get the code

Clone the repository and open the project directory.

In [ ]:
!git clone https://github.com/bondarenkoania/dla_asr.git
%cd dla_asr

### 2. Set up the environment

Create a Python 3.11 environment for the project's dependencies. All project commands below use this environment.

In [ ]:
%pip install uv

!uv venv --python 3.11 /content/asr-venv

!uv pip install --python /content/asr-venv/bin/python -r requirements.txt

### 3. Download the checkpoint and LM

Download the Conformer model checkpoint from a public Hugging Face repository and LM file for decoding.

In [ ]:
!/content/asr-venv/bin/python download_resources.py

### 4. Download your dataset

Paste a direct download link to a ZIP archive with your data. The archive should contain `audio/` and, optionally, `transcriptions/` at its root:

```text
audio/
  utterance_1.wav
  utterance_2.flac
transcriptions/
  utterance_1.txt
  utterance_2.txt
```

If transcriptions are provided, every audio file must have a corresponding TXT file with the same filename.

In [ ]:
dataset_url = "PASTE_YOUR_DATASET_ZIP_URL_HERE"

!wget -O dataset.zip "{dataset_url}"
!mkdir -p data/demo/custom
!unzip -oq dataset.zip -d data/demo/custom


### 5. Run inference

Run the command below with Conformer Medium and LM beam search. To compare all three decoders, replace `decoder=beam_lm` with `decoder=compare`.

Predictions are saved as TXT files with the original utterance IDs in `data/saved/custom_predictions/test/`.

In [ ]:
custom_dir = "data/demo/custom"

In [ ]:
!/content/asr-venv/bin/python inference.py datasets=custom decoder=beam_lm \
  datasets.test.custom_dir="{custom_dir}" \
  inferencer.from_pretrained=saved/pretrained/model_best.pth \
  decoder.BeamSearchLM.path_to_lm=data/lm/librispeech_3gram.arpa \
  inferencer.save_path=custom_predictions 

If your dataset contains fewer than 10 audio files, add `dataloader.batch_size=1` to the inference command.

### 6. Calculate metrics

You can compare references and predictions using `calc_metrics.py`. CER and WER are reported as percentages.

With `decoder=compare`, predictions are saved in `Argmax/`, `BeamSearch/`, and `BeamSearchLM/` subdirectories. Replace the prediction path below with the desired subdirectory, for example `data/saved/custom_predictions/test/BeamSearchLM`.

In [ ]:
!/content/asr-venv/bin/python calc_metrics.py data/demo/custom/transcriptions data/saved/custom_predictions/test
